<a href="https://colab.research.google.com/github/radwaelreedy/DEPI_Tech_Journey/blob/main/Assignment__SVM_NB/Assignment__SVM_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget http://qwone.com/~jason/20Newsgroups/20news-bydate.tar.gz
!tar -xzf 20news-bydate.tar.gz

--2026-01-03 17:06:45--  http://qwone.com/~jason/20Newsgroups/20news-bydate.tar.gz
Resolving qwone.com (qwone.com)... 173.48.205.131
Connecting to qwone.com (qwone.com)|173.48.205.131|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14464277 (14M) [application/x-gzip]
Saving to: ‘20news-bydate.tar.gz’

20news-bydate.tar.g 100%[===================>]  13.79M  9.42MB/s    in 1.5s    

2026-01-03 17:06:47 (9.42 MB/s) - ‘20news-bydate.tar.gz’ saved [14464277/14464277]



In [ ]:
from sklearn.datasets import load_files
categories = ['sci.med', 'sci.space', 'comp.graphics', 'rec.sport.baseball']

train_path = '20news-bydate-train'
data = load_files(train_path, categories=categories, encoding='latin1')
X_text = data.data
y = data.target

In [ ]:
X_text

['From: baalke@kelvin.jpl.nasa.gov (Ron Baalke)\nSubject: JPL\'s VLBI Project Meets with International Space Agencies\nOrganization: Jet Propulsion Laboratory\nLines: 112\nDistribution: world\nNNTP-Posting-Host: kelvin.jpl.nasa.gov\nKeywords: VLBI, JPL\nNews-Software: VAX/VMS VNEWS 1.41    \n\nFrom the "JPL Universe"\nApril 23, 1993\n\nVLBI project meets with international space agencies\n\nBy Ed McNevin\n     Members of JPL\'s Space Very Long Baseline Interferometry\n(VLBI) project team recently concluded a week-long series of\nmeetings with officials from Russia and Japan.\n     The meetings were part of "Space VLBI Week" held at JPL in\nearly March and were intended to maintain cooperation between\ninternational space agencies participating in the development of\nthe U.S. Space VLBI Project, a recently approved JPL flight\nproject set for launch in 1995.\n     U.S. Space VLBI will utilize two Earth-orbiting spacecraft\n-- the Japanese VSOP (VLBI Space Observing Program) satellite\nw

In [ ]:
import re

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

# Apply cleaning
X_clean = [clean_text(text) for text in X_text]

In [ ]:
X_clean

['from baalkekelvinjplnasagov ron baalke subject jpls vlbi project meets with international space agencies organization jet propulsion laboratory lines distribution world nntppostinghost kelvinjplnasagov keywords vlbi jpl newssoftware vaxvms vnews from the jpl universe april vlbi project meets with international space agencies by ed mcnevin members of jpls space very long baseline interferometry vlbi project team recently concluded a weeklong series of meetings with officials from russia and japan the meetings were part of space vlbi week held at jpl in early march and were intended to maintain cooperation between international space agencies participating in the development of the us space vlbi project a recently approved jpl flight project set for launch in us space vlbi will utilize two earthorbiting spacecraft the japanese vsop vlbi space observing program satellite with its meter radio telescope and a russian radioastron meter satellite both spacecraft will team up with groundbase

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform your text data
X_tfidf = vectorizer.fit_transform(X_clean)  # X_clean from previous cleaning step

print(f"Shape: {X_tfidf.shape}")
print(f"Type: {type(X_tfidf)}")  # Sparse matrix
print(f"First vector:\n{X_tfidf[0].toarray()}")

Shape: (2368, 36225)
Type: <class 'scipy.sparse._csr.csr_matrix'>
First vector:
[[0. 0. 0. ... 0. 0. 0.]]


In [ ]:
import numpy as np
from collections import Counter

# Check class distribution
class_counts = Counter(y)
print("Class distribution:")
for cls, count in class_counts.items():
    print(f"Class {cls}: {count} samples ({count/len(y)*100:.1f}%)")

# Or using numpy
unique, counts = np.unique(y, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count} samples")

Class distribution:
Class 3: 593 samples (25.0%)
Class 1: 597 samples (25.2%)
Class 2: 594 samples (25.1%)
Class 0: 584 samples (24.7%)
Class 0: 584 samples
Class 1: 597 samples
Class 2: 594 samples
Class 3: 593 samples


In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split - maintains class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # This maintains proportions
)

# Verify distribution
print("Train distribution:", Counter(y_train))
print("Test distribution:", Counter(y_test))

Train distribution: Counter({np.int64(1): 478, np.int64(2): 475, np.int64(3): 474, np.int64(0): 467})
Test distribution: Counter({np.int64(1): 119, np.int64(2): 119, np.int64(3): 119, np.int64(0): 117})


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Create and train NB classifier
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Make predictions
y_pred = nb.predict(X_test)

# Check accuracy
accuracy = nb.score(X_test, y_test)
print(f"Accuracy: {accuracy:.3f}")

Accuracy: 0.964


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Get predictions first
y_pred = nb.predict(X_test)

# Individual metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print(" Individual Metrics : ")
print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-Score:  {f1:.3f}")

# Detailed classification report
print("\n=== Classification Report : ")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("\n=== Confusion Matrix : ")
cm = confusion_matrix(y_test, y_pred)
print(cm)

 Individual Metrics : 
Accuracy:  0.964
Precision: 0.965
Recall:    0.964
F1-Score:  0.964

=== Classification Report : 
              precision    recall  f1-score   support

           0       0.97      0.92      0.95       117
           1       0.98      1.00      0.99       119
           2       0.97      0.97      0.97       119
           3       0.93      0.97      0.95       119

    accuracy                           0.96       474
   macro avg       0.96      0.96      0.96       474
weighted avg       0.96      0.96      0.96       474


=== Confusion Matrix : 
[[108   2   2   5]
 [  0 119   0   0]
 [  0   0 115   4]
 [  3   0   1 115]]


In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Train NB
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Get log priors
print("Log priors (log P(y)):")
for i, log_prior in enumerate(nb.class_log_prior_):
    print(f"  Class {i}: {log_prior:.3f}")

# Convert to regular probabilities
print("\nClass probabilities P(y):")
for i, log_prior in enumerate(nb.class_log_prior_):
    prob = np.exp(log_prior)
    print(f"  Class {i}: {prob:.3f}")

Log priors (log P(y)):
  Class 0: -1.400
  Class 1: -1.377
  Class 2: -1.383
  Class 3: -1.385

Class probabilities P(y):
  Class 0: 0.247
  Class 1: 0.252
  Class 2: 0.251
  Class 3: 0.250


In [ ]:
from sklearn.svm import LinearSVC

# Create and train SVM
svm = LinearSVC(random_state=42)
svm.fit(X_train, y_train)

# Evaluate
train_score = svm.score(X_train, y_train)
test_score = svm.score(X_test, y_test)

print(f"Training accuracy: {train_score:.3f}")
print(f"Testing accuracy: {test_score:.3f}")

Training accuracy: 1.000
Testing accuracy: 0.981


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Get predictions
y_pred = svm.predict(X_test)

# 1. Accuracy
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

# 2. Full report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 3. Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.981

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       117
           1       0.98      1.00      0.99       119
           2       1.00      0.96      0.98       119
           3       0.99      0.97      0.98       119

    accuracy                           0.98       474
   macro avg       0.98      0.98      0.98       474
weighted avg       0.98      0.98      0.98       474

Confusion Matrix:
[[116   1   0   0]
 [  0 119   0   0]
 [  3   1 114   1]
 [  3   0   0 116]]


1. Why does Naive Bayes perform well despite violating independence?

    Naive Bayes works well because text features often have weak dependencies, and the model benefits from the bias-variance trade-off.
2. Why does SVM usually outperform NB on TF-IDF features?

    SVM outperforms NB because TF-IDF creates correlated features, and SVM handles this better with its margin optimization.
3. Why does increasing `C` risk overfitting?

    Increasing C risks overfitting because it reduces regularization, allowing the model to fit noise in training data.
4. Why is probability calibration expensive for SVM?

    Probability calibration is expensive because SVM needs Platt scaling with cross-validation, adding computational overhead.
5. Why does NB scale better to massive datasets?

    NB scales better because it requires only one pass through data and has O(N) complexity with simple probability updates.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Minimal GridSearch
param_grid = {'C': [0.1, 1, 10]}
grid = GridSearchCV(LinearSVC(), param_grid, cv=3, scoring='accuracy')
grid.fit(X_train, y_train)

print(f"Best C: {grid.best_params_['C']}")
print(f"Best CV accuracy: {grid.best_score_:.3f}")

# Use best model
best_svm = grid.best_estimator_
print(f"Test accuracy: {best_svm.score(X_test, y_test):.3f}")

Best C: 1
Best CV accuracy: 0.973
Test accuracy: 0.981
